# LeetCode 2330 — Valid Palindrome IV bằng Python

Link đề: https://leetcode.com/problems/valid-palindrome-iv/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def makePalindrome(self, s: str) -> bool:
        ...
```

## 1. Mục tiêu học

1. Nhận ra chuỗi là palindrome ⇔ mọi **cặp đối xứng** `(i, n−1−i)` bằng nhau; số thao tác tối thiểu = **số cặp lệch** `m`.
2. Đếm `m` bằng **hai con trỏ** từ hai đầu vào giữa (không đếm hai lần một cặp).
3. Phân tích chữ "**đúng** một hoặc hai thao tác": khi `m = 0` vẫn phải thực hiện thao tác mà giữ được palindrome — luôn làm được (đổi ký tự giữa nếu `n` lẻ, đổi cả một cặp nếu `n` chẵn).
4. Kiểm chứng bằng BFS vét cạn trên mọi chuỗi đạt được sau đúng 1 hoặc 2 thao tác.
5. Tổng quát hóa: "đúng `k` thao tác" và các biến thể (xóa ký tự — LC 680/1216).

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ thường. Mỗi thao tác: đổi một ký tự bất kỳ thành **một ký tự khác**. Trả về `true` nếu có thể biến `s` thành palindrome sau **đúng một hoặc hai** thao tác.

**Ví dụ 1:** `s = "abcdba"` → `true` (đổi `s[2]` thành `'d'`: `"abddba"`).

**Ví dụ 2:** `s = "aa"` → `true` (đổi `s[0]` thành `'b'`, rồi `s[1]` thành `'b'`: `"bb"`).

**Ví dụ 3:** `s = "abcdef"` → `false`.

**Ràng buộc:** `1 ≤ len(s) ≤ 10⁵`; chỉ chữ thường.

In [1]:
from collections import deque
from itertools import product
import random, string, time

examples = [("abcdba", True), ("aa", True), ("abcdef", False)]
for s, exp in examples:
    pairs = [(s[i], s[len(s) - 1 - i]) for i in range(len(s) // 2)]
    print(f"{s!r}: các cặp đối xứng {pairs} -> kỳ vọng {exp}")

'abcdba': các cặp đối xứng [('a', 'a'), ('b', 'b'), ('c', 'd')] -> kỳ vọng True
'aa': các cặp đối xứng [('a', 'a')] -> kỳ vọng True
'abcdef': các cặp đối xứng [('a', 'f'), ('b', 'e'), ('c', 'd')] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 Palindrome qua các cặp đối xứng

`s` là palindrome ⇔ `s[i] = s[n−1−i]` với mọi `0 ≤ i < n/2`. Ký tự giữa (khi `n` lẻ) không có cặp — luôn "khớp" với chính nó.

### 3.2 Một thao tác sửa được tối đa một cặp

Đổi `s[i]` chỉ ảnh hưởng cặp chứa `i`. Một cặp lệch cần ít nhất một thao tác (đổi một bên thành bên kia) ⇒ số thao tác tối thiểu đúng bằng `m` = số cặp lệch.

In [2]:
def mismatches(s):
    return sum(s[i] != s[-1 - i] for i in range(len(s) // 2))

for s, _ in examples:
    print(repr(s), "số cặp lệch =", mismatches(s))

'abcdba' số cặp lệch = 1
'aa' số cặp lệch = 0
'abcdef' số cặp lệch = 3


### 3.3 Hai con trỏ

`i = 0`, `j = n − 1`; trong khi `i < j`: so `s[i]`, `s[j]`, rồi `i += 1`, `j −= 1`. Mỗi cặp được xét đúng một lần, O(n/2).

### 3.4 Chuỗi đảo `s[::-1]` và vì sao so trực tiếp lại đếm **gấp đôi**

`Σ_i [s[i] ≠ s[::-1][i]]` duyệt cả `n` vị trí ⇒ mỗi cặp lệch được đếm **hai lần** (ở `i` và ở `n − 1 − i`).

In [3]:
s = "abcdef"
print("so với chuỗi đảo:", sum(a != b for a, b in zip(s, s[::-1])), "| số cặp lệch:", mismatches(s))

so với chuỗi đảo: 6 | số cặp lệch: 3


## 4. Ý tưởng nền tảng / trực giác

Gọi `m` = số cặp lệch.

- `m ≥ 3`: cần ≥ 3 thao tác ⇒ `false`.
- `m = 1`: một thao tác sửa cặp đó ⇒ `true` (đúng 1 thao tác).
- `m = 2`: hai thao tác, mỗi thao tác sửa một cặp ⇒ `true`.
- `m = 0`: đã là palindrome nhưng **bắt buộc** thực hiện 1 hoặc 2 thao tác:
  - `n` lẻ: đổi ký tự giữa (1 thao tác) — vẫn palindrome;
  - `n` chẵn: đổi **cả hai** ký tự của một cặp thành cùng một chữ mới (2 thao tác) — vẫn palindrome (ví dụ 2: `"aa" → "ba" → "bb"`).
  ⇒ luôn `true`.

⇒ Đáp án: `m ≤ 2`.

Tính tay ví dụ 1 `"abcdba"`: cặp `(a, a)`, `(b, b)`, `(c, d)` ⇒ `m = 1` ⇒ `true`. Ví dụ 3 `"abcdef"`: `(a, f)`, `(b, e)`, `(c, d)` ⇒ `m = 3` ⇒ `false`.

In [4]:
def witness(s):
    # trả về dãy thao tác (vị trí, ký tự mới) dài 1 hoặc 2 biến s thành palindrome, hoặc None
    n = len(s); t = list(s); ops = []
    for i in range(n // 2):
        if t[i] != t[n - 1 - i]: ops.append((n - 1 - i, t[i])); t[n - 1 - i] = t[i]
    if len(ops) > 2: return None
    if not ops:
        if n % 2: c = "b" if t[n // 2] == "a" else "a"; ops = [(n // 2, c)]
        else: c = "b" if t[0] == "a" else "a"; ops = [(0, c), (n - 1, c)]
    return ops

for s, _ in examples + [("aba", None), ("abba", None)]:
    ops = witness(s); t = list(s)
    if ops:
        for p, c in ops: assert t[p] != c; t[p] = c
    print(repr(s), "->", ops, "->", "".join(t) if ops else None)

'abcdba' -> [(3, 'c')] -> abccba
'aa' -> [(0, 'b'), (1, 'b')] -> bb
'abcdef' -> None -> None
'aba' -> [(1, 'a')] -> aaa
'abba' -> [(0, 'b'), (3, 'b')] -> bbbb


## 5. Cách 1 — Brute Force / baseline

### 5.1 BFS mọi chuỗi đạt được sau **đúng** 1 hoặc 2 thao tác (bảng chữ cái thu nhỏ)

Với chuỗi ngắn, liệt kê mọi kết quả sau 1 thao tác (đổi một vị trí sang chữ **khác**) và sau 2 thao tác; kiểm tra có palindrome không. Dùng bảng chữ cái gồm các chữ trong `s` cộng thêm 2 chữ mới — đủ đại diện cho mọi lựa chọn (các chữ không xuất hiện là "như nhau").

In [5]:
def brute(s):
    letters = sorted(set(s))
    extra = [c for c in string.ascii_lowercase if c not in letters][:2]
    alpha = letters + extra
    def step(t):
        for i in range(len(t)):
            for c in alpha:
                if c != t[i]: yield t[:i] + c + t[i + 1:]
    one = set(step(s))
    if any(t == t[::-1] for t in one): return True
    for t in one:
        for u in step(t):
            if u == u[::-1]: return True
    return False

for s, exp in examples:
    print(brute(s), "OK" if brute(s) == exp else "SAI")

True OK
True OK
False OK


### 5.1b Bảng đầy đủ các trường hợp theo `m` và tính chẵn lẻ của `n`

| m | n lẻ | n chẵn | cách làm |
|---|---|---|---|
| 0 | ✔ (1 thao tác: ký tự giữa) | ✔ (2 thao tác: một cặp) | "đốt" thao tác mà vẫn giữ palindrome |
| 1 | ✔ | ✔ | sửa cặp lệch |
| 2 | ✔ | ✔ | sửa hai cặp lệch |
| ≥ 3 | ✘ | ✘ | cần ≥ 3 thao tác |

Kiểm chứng bảng bằng BFS trên mọi chuỗi ngắn trên bảng chữ cái `{a, b}`:

In [6]:
table = {}
for n in range(1, 7):
    for t in product("ab", repeat=n):
        s = "".join(t); m = mismatches(s)
        table.setdefault((min(m, 3), n % 2), set()).add(brute(s))
for (m, par), res in sorted(table.items()):
    print(f"m = {m if m < 3 else '>=3'}, n {'lẻ ' if par else 'chẵn'}: BFS cho {res}")

m = 0, n chẵn: BFS cho {True}
m = 0, n lẻ : BFS cho {True}
m = 1, n chẵn: BFS cho {True}
m = 1, n lẻ : BFS cho {True}
m = 2, n chẵn: BFS cho {True}
m = 2, n lẻ : BFS cho {True}
m = >=3, n chẵn: BFS cho {False}


Ô `m ≥ 3, n lẻ` không xuất hiện vì cần `n ≥ 7` (ba cặp và một ký tự giữa). Bổ sung riêng:

In [7]:
cases7 = ["".join(t) for t in product("ab", repeat=7) if mismatches("".join(t)) >= 3]
print(len(cases7), "chuỗi độ dài 7 có m = 3; BFS cho", {brute(s) for s in cases7[:20]}, "(thử 20 chuỗi đầu)")

16 chuỗi độ dài 7 có m = 3; BFS cho {False} (thử 20 chuỗi đầu)


O((n · |Σ|)²) chuỗi, mỗi chuỗi O(n) để kiểm tra — chỉ dùng cho `n ≤ 7`.

### 5.2 Baseline 2 — So với chuỗi đảo, đếm vị trí khác rồi chia 2 — O(n), bộ nhớ O(n)

In [8]:
def reverse_compare(s):
    diff = sum(a != b for a, b in zip(s, s[::-1]))
    return diff // 2 <= 2

for s, exp in examples:
    print(reverse_compare(s), "OK" if reverse_compare(s) == exp else "SAI")

True OK
True OK
False OK


Đúng (vì `diff = 2m`), nhưng tạo bản sao đảo O(n) bộ nhớ và duyệt gấp đôi. **Nút thắt** không đáng kể; hai con trỏ chỉ là cách gọn và có thể **dừng sớm** khi `m > 2`.

## 6. Ý tưởng thuật toán / bước đột phá

```
i, j, m = 0, n − 1, 0
while i < j:
    if s[i] != s[j]: m += 1; if m > 2: return False   # dừng sớm
    i += 1; j −= 1
return True
```

- O(n) thời gian (dừng sớm ngay khi gặp cặp lệch thứ ba), O(1) bộ nhớ.
- Trường hợp `m = 0` không cần xử lý riêng — đáp án là `true` (mục 4).

### 6.1 Nếu thao tác được phép "đổi thành chính nó" thì sao?

Đề nói "đổi thành **ký tự khác**". Nếu cho phép đổi thành chính nó, trường hợp `m = 0` trở nên tầm thường (thao tác không làm gì). Đáp án vẫn là `m ≤ 2` — tức là chi tiết "ký tự khác" không làm thay đổi kết quả, nhưng làm thay đổi **lập luận** (phải chỉ ra cách "đốt" thao tác hợp lệ). Đây là lý do mục 4 phải xét riêng `n` lẻ và `n` chẵn.

### 6.2 Vì sao bảng chữ cái thu nhỏ trong BFS là đủ?

Mọi chữ không xuất hiện trong `s` đều "đối xứng" với nhau: hoán đổi tên chúng biến một dãy thao tác hợp lệ thành dãy hợp lệ khác cho cùng tính palindrome. Với tối đa 2 thao tác, ta cần nhiều nhất 2 chữ mới khác nhau ⇒ bảng chữ cái `chữ trong s ∪ {2 chữ mới}` đại diện cho mọi khả năng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai con trỏ đếm hết (cách doocs)

In [9]:
def two_pointers(s):
    i, j = 0, len(s) - 1; cnt = 0
    while i < j:
        cnt += s[i] != s[j]
        i, j = i + 1, j - 1
    return cnt <= 2

for s, exp in examples:
    print(two_pointers(s), "OK" if two_pointers(s) == exp else "SAI")

True OK
True OK
False OK


### 7.2 Cách 3 — Hai con trỏ dừng sớm

In [10]:
def early_exit(s):
    i, j = 0, len(s) - 1; cnt = 0
    while i < j:
        if s[i] != s[j]:
            cnt += 1
            if cnt > 2: return False
        i += 1; j -= 1
    return True

for s, exp in examples:
    print(early_exit(s), "OK" if early_exit(s) == exp else "SAI")

True OK
True OK
False OK


### 7.3 Cách 4 — Một dòng với `zip` nửa đầu và nửa sau đảo

In [11]:
def zip_halves(s):
    h = len(s) // 2
    return sum(a != b for a, b in zip(s[:h], reversed(s[-h:] if h else ""))) <= 2

for s, exp in examples:
    print(zip_halves(s), "OK" if zip_halves(s) == exp else "SAI")

True OK
True OK
False OK


Lưu ý `s[-0:]` là **cả chuỗi** (vì `-0 == 0`), nên phải xử lý `h = 0` (chuỗi dài 1) riêng — một bẫy nhỏ của cắt chuỗi với chỉ số âm.

### 7.4 Cách 5 — Xây dựng nghiệm cụ thể (trả về các thao tác)

`witness(s)` ở mục 4 vừa quyết định vừa trả về dãy thao tác; `makePalindrome = witness(s) is not None`.

In [12]:
def constructive(s):
    return witness(s) is not None

for s, exp in examples:
    print(constructive(s), "OK" if constructive(s) == exp else "SAI", witness(s))

True OK [(3, 'c')]
True OK [(0, 'b'), (1, 'b')]
False OK None


## 8. TRACE từng bước

### 8.1 Trace hai con trỏ

In [13]:
def trace(s):
    i, j = 0, len(s) - 1; cnt = 0
    print(f"s = {s!r}")
    while i < j:
        same = s[i] == s[j]; cnt += not same
        print(f"  i={i} j={j}: {s[i]} vs {s[j]} -> {'khớp' if same else 'LỆCH'} (cnt = {cnt})")
        i, j = i + 1, j - 1
    print("  ->", cnt <= 2)

for s, _ in examples: trace(s)

s = 'abcdba'
  i=0 j=5: a vs a -> khớp (cnt = 0)
  i=1 j=4: b vs b -> khớp (cnt = 0)
  i=2 j=3: c vs d -> LỆCH (cnt = 1)
  -> True
s = 'aa'
  i=0 j=1: a vs a -> khớp (cnt = 0)
  -> True
s = 'abcdef'
  i=0 j=5: a vs f -> LỆCH (cnt = 1)
  i=1 j=4: b vs e -> LỆCH (cnt = 2)
  i=2 j=3: c vs d -> LỆCH (cnt = 3)
  -> False


### 8.2 BFS kiểm chứng chữ "đúng" cho chuỗi đã là palindrome

In [14]:
for s in ["a", "aa", "aba", "abba", "racecar"]:
    print(f"{s!r}: palindrome sẵn, BFS (đúng 1 hoặc 2 thao tác) -> {brute(s)}, thao tác mẫu {witness(s)}")

'a': palindrome sẵn, BFS (đúng 1 hoặc 2 thao tác) -> True, thao tác mẫu [(0, 'b')]
'aa': palindrome sẵn, BFS (đúng 1 hoặc 2 thao tác) -> True, thao tác mẫu [(0, 'b'), (1, 'b')]
'aba': palindrome sẵn, BFS (đúng 1 hoặc 2 thao tác) -> True, thao tác mẫu [(1, 'a')]
'abba': palindrome sẵn, BFS (đúng 1 hoặc 2 thao tác) -> True, thao tác mẫu [(0, 'b'), (3, 'b')]
'racecar': palindrome sẵn, BFS (đúng 1 hoặc 2 thao tác) -> True, thao tác mẫu [(3, 'a')]


Mọi palindrome đều cho `true`: độ dài lẻ đổi ký tự giữa; độ dài chẵn đổi một cặp thành chữ mới.

### 8.3 Vì sao thao tác thứ hai không "làm hỏng" cặp đã sửa?

Trong cách xây dựng, hai thao tác (khi `m = 2`) nằm ở **hai cặp khác nhau**, nên không ảnh hưởng nhau. Khi `m = 0` và `n` chẵn, hai thao tác nằm ở **cùng một cặp** `(0, n−1)` và đưa cả hai về cùng chữ mới — sau thao tác thứ nhất chuỗi tạm thời **không** palindrome (`"ba"`), chỉ sau thao tác thứ hai mới lại là palindrome (`"bb"`). Đề chỉ yêu cầu kết quả **cuối** là palindrome, nên trạng thái trung gian không quan trọng.

In [15]:
s = "abba"; t = list(s)
for p, c in witness(s):
    t[p] = c; cur = "".join(t)
    print(f"đổi vị trí {p} thành {c!r}: {cur!r} -> palindrome? {cur == cur[::-1]}")

đổi vị trí 0 thành 'b': 'bbba' -> palindrome? False
đổi vị trí 3 thành 'b': 'bbbb' -> palindrome? True


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def makePalindrome(self, s: str) -> bool:
        i, j = 0, len(s) - 1
        mismatched = 0
        while i < j:
            if s[i] != s[j]:
                mismatched += 1               # mỗi cặp lệch cần đúng 1 thao tác
                if mismatched > 2:
                    return False
            i += 1
            j -= 1
        return True                           # m = 0, 1, 2 đều làm được với đúng 1 hoặc 2 thao tác

In [17]:
sol = Solution()
for s, exp in examples:
    got = sol.makePalindrome(s)
    print(repr(s), got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

'abcdba' True | kỳ vọng True | OK
'aa' True | kỳ vọng True | OK
'abcdef' False | kỳ vọng False | OK


## 10. Vì sao thuật toán đúng

Gọi `P = {(i, n−1−i) : 0 ≤ i < ⌊n/2⌋}` các cặp đối xứng, `m` = số cặp lệch.

**Chặn dưới.** Mỗi thao tác đổi một vị trí, thuộc tối đa một cặp ⇒ sau `t` thao tác, tối đa `t` cặp lệch được sửa (và có thể làm lệch cặp khác). Để kết quả là palindrome cần mọi cặp khớp ⇒ `t ≥ m`. Vậy `m ≥ 3` ⇒ không thể với `t ≤ 2` ⇒ `false`.

**Đạt được khi `m ∈ {1, 2}`.** Với mỗi cặp lệch `(i, j)`, đổi `s[j]` thành `s[i]` (khác `s[j]` nên là thao tác hợp lệ) — `m` thao tác, đúng 1 hoặc 2.

**Đạt được khi `m = 0`.** `n` lẻ: đổi `s[n//2]` sang chữ khác — 1 thao tác, ký tự giữa không có cặp ⇒ vẫn palindrome. `n` chẵn (`n ≥ 2`): chọn cặp `(0, n−1)`, `s[0] = s[n−1] = x`; đổi cả hai sang `y ≠ x` — 2 thao tác, mỗi thao tác đổi sang chữ **khác** ký tự hiện tại; cặp vẫn khớp, các cặp khác không đổi.

⇒ `makePalindrome(s) ⇔ m ≤ 2`. Hai con trỏ đếm đúng `m` (mỗi cặp xét một lần). ∎

**Về số thao tác tối thiểu.** Lập luận trên cũng chứng minh rằng số thao tác **tối thiểu** để biến `s` thành palindrome (nếu cho phép 0 thao tác) đúng bằng `m`: chặn dưới `t ≥ m`, và cách "đổi một bên của mỗi cặp lệch" đạt đúng `m`. Bài gốc chỉ là câu hỏi `m ≤ 2` cộng thêm việc xử lý `m = 0` cho chữ "đúng". Hai con trỏ dừng sớm có thể trả lời câu hỏi "`m ≤ k`?" trong thời gian tỉ lệ với vị trí của cặp lệch thứ `k + 1` — hữu ích khi chuỗi dài và lệch nhiều ngay từ đầu.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS vét cạn | O((n·|Σ|)² · n) | O((n·|Σ|)²) |
| 1b. So với chuỗi đảo | O(n) | O(n) |
| 2. Hai con trỏ (doocs) | O(n) | O(1) |
| 3. Hai con trỏ dừng sớm | O(n), thường nhanh hơn | O(1) |
| 4. `zip` nửa chuỗi | O(n) | O(n) (lát cắt) |
| 5. Xây dựng nghiệm | O(n) | O(n) |

In [18]:
rng = random.Random(2330)
half = "".join(rng.choice(string.ascii_lowercase) for _ in range(50000))
cases = {"palindrome 10^5": half + half[::-1], "lệch ở đầu": "ab" + "c" * 99996 + "de",
         "ngẫu nhiên": "".join(rng.choice("ab") for _ in range(10**5))}
for label, s in cases.items():
    res = []
    for name, f in [("reverse_compare", reverse_compare), ("two_pointers", two_pointers), ("early_exit", early_exit),
                    ("zip_halves", zip_halves), ("Solution", Solution().makePalindrome)]:
        t0 = time.perf_counter(); res.append(f(s)); dt = time.perf_counter() - t0
        print(f"{label:<16} {name:<16} {dt*1000:7.2f} ms -> {res[-1]}")
    assert len(set(res)) == 1

palindrome 10^5  reverse_compare     3.16 ms -> True
palindrome 10^5  two_pointers        2.66 ms -> True
palindrome 10^5  early_exit          2.30 ms -> True
palindrome 10^5  zip_halves          1.81 ms -> True
palindrome 10^5  Solution            2.22 ms -> True
lệch ở đầu       reverse_compare     3.11 ms -> True
lệch ở đầu       two_pointers        2.68 ms -> True
lệch ở đầu       early_exit          2.67 ms -> True
lệch ở đầu       zip_halves          1.58 ms -> True
lệch ở đầu       Solution            2.26 ms -> True
ngẫu nhiên       reverse_compare     3.71 ms -> False
ngẫu nhiên       two_pointers        3.57 ms -> False
ngẫu nhiên       early_exit          0.00 ms -> False
ngẫu nhiên       zip_halves          1.97 ms -> False
ngẫu nhiên       Solution            0.00 ms -> False


Với chuỗi ngẫu nhiên, cặp lệch thứ ba xuất hiện gần như ngay ở đầu ⇒ `early_exit` gần như tức thì; các cách khác duyệt hết. Với palindrome dài thì mọi cách đều phải duyệt hết nửa chuỗi — không có cách nào nhanh hơn O(n) vì phải đọc mọi ký tự.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng — đối chiếu lại đề gốc nếu có Premium)

In [19]:
ALL = {"brute": brute, "reverse_compare": reverse_compare, "two_pointers": two_pointers, "early_exit": early_exit,
       "zip_halves": zip_halves, "constructive": constructive, "Solution": Solution().makePalindrome}
edge = examples + [
    ("a", True),                   # một ký tự: đổi nó, vẫn palindrome
    ("ab", True),                  # 1 cặp lệch
    ("aba", True),                 # palindrome lẻ: đổi ký tự giữa
    ("abcd", True),                # 2 cặp lệch
    ("abcde", True),               # 2 cặp lệch (c ở giữa)
    ("abcdea", True),              # (a,a), (b,e), (c,d): 2 cặp lệch
    ("abcdxyz", False),            # 3 cặp lệch: (a,z), (b,y), (c,x)
    ("aabbcc", True),              # (a,c), (a,c), (b,b): chỉ 2 cặp lệch
]
for s, exp in edge:
    ref = brute(s)
    for name, f in ALL.items():
        assert f(s) == ref, (name, s)
    print(f"{s!r:<10} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

'abcdba'   -> True | ghi trong bảng: True | khớp: True
'aa'       -> True | ghi trong bảng: True | khớp: True
'abcdef'   -> False | ghi trong bảng: False | khớp: True
'a'        -> True | ghi trong bảng: True | khớp: True
'ab'       -> True | ghi trong bảng: True | khớp: True
'aba'      -> True | ghi trong bảng: True | khớp: True
'abcd'     -> True | ghi trong bảng: True | khớp: True
'abcde'    -> True | ghi trong bảng: True | khớp: True
'abcdea'   -> True | ghi trong bảng: True | khớp: True
'abcdxyz'  -> False | ghi trong bảng: False | khớp: True
'aabbcc'   -> True | ghi trong bảng: True | khớp: True


Case `"aabbcc"`: các cặp là `(a, c)`, `(a, c)`, `(b, b)` ⇒ chỉ **2** cặp lệch ⇒ `true`. Dễ đoán nhầm `false` nếu nhìn "ba nhóm chữ khác nhau" thay vì đếm **cặp đối xứng** — cột `khớp:` là lưới an toàn cho loại nhầm lẫn này.

### 12.2 Đối chiếu ngẫu nhiên với BFS (seed cố định)

In [20]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 6); s = "".join(rng.choice("abc") for _ in range(n))
    ref = brute(s)
    for name, f in ALL.items():
        assert f(s) == ref, (name, s)
print("400 chuỗi ngẫu nhiên OK")

400 chuỗi ngẫu nhiên OK


### 12.3 Tính chất: nghiệm xây dựng hợp lệ; thêm một cặp lệch làm `m` tăng đúng 1

In [21]:
rng = random.Random(7)
for _ in range(2000):
    n = rng.randint(1, 40); s = "".join(rng.choice("abcd") for _ in range(n))
    ops = witness(s)
    assert (ops is not None) == Solution().makePalindrome(s)
    if ops:
        t = list(s)
        for p, c in ops: assert t[p] != c; t[p] = c
        assert t == t[::-1] and 1 <= len(ops) <= 2
    assert mismatches("x" + s + "y") == mismatches(s) + 1
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: độ dài `10⁵` với đúng 0, 1, 2, 3 cặp lệch ở vị trí xa nhau

In [22]:
base = list("a" * 10**5)
for m, expect in [(0, True), (1, True), (2, True), (3, False)]:
    t = base[:]
    for q in range(m): t[q * 30000] = "z"                     # mỗi vị trí 'z' tạo một cặp lệch
    s = "".join(t)
    assert mismatches(s) == m and Solution().makePalindrome(s) == expect == two_pointers(s)
print("độ dài 10^5 với m = 0..3 OK")

độ dài 10^5 với m = 0..3 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So với chuỗi đảo và so sánh trực tiếp `≤ 2` (đếm gấp đôi)

In [23]:
def wrong_double(s):
    return sum(a != b for a, b in zip(s, s[::-1])) <= 2          # SAI: mỗi cặp lệch đếm 2 lần

for s in ["abcdba", "abcd", "abcde"]:
    print("bản sai :", wrong_double(s), "| bản đúng:", Solution().makePalindrome(s), "cho", repr(s))

bản sai : True | bản đúng: True cho 'abcdba'
bản sai : False | bản đúng: True cho 'abcd'
bản sai : False | bản đúng: True cho 'abcde'


**Vì sao sai:** ví dụ 1 **trùng hợp** đúng (1 cặp lệch ⇒ `diff = 2`). `"abcd"` có 2 cặp lệch ⇒ `diff = 4 > 2` nhưng đáp án `true`. **Sửa:** chia 2, hoặc chỉ duyệt nửa chuỗi.

### Lỗi 2 — Trả `false` khi chuỗi đã là palindrome (hiểu sai chữ "đúng")

In [24]:
def wrong_exactly(s):
    m = mismatches(s)
    return 1 <= m <= 2                                            # SAI: m = 0 vẫn làm được

for s, _ in examples + [("aba", None)]:
    print("bản sai :", wrong_exactly(s), "| bản đúng:", Solution().makePalindrome(s), "cho", repr(s))

bản sai : True | bản đúng: True cho 'abcdba'
bản sai : False | bản đúng: True cho 'aa'
bản sai : False | bản đúng: False cho 'abcdef'
bản sai : False | bản đúng: True cho 'aba'


**Vì sao sai:** ví dụ 2 (`"aa"`) chính là trường hợp này — đề cho đáp án `true` bằng cách đổi cả hai ký tự. **Sửa:** `m ≤ 2`.

### Lỗi 3 — Chỉ cho phép **một** thao tác (`m ≤ 1`)

In [25]:
def wrong_one(s):
    return mismatches(s) <= 1                                     # SAI

for s in ["abcdba", "abcd", "abcdef"]:
    print("bản sai :", wrong_one(s), "| bản đúng:", Solution().makePalindrome(s), "cho", repr(s))

bản sai : True | bản đúng: True cho 'abcdba'
bản sai : False | bản đúng: True cho 'abcd'
bản sai : False | bản đúng: False cho 'abcdef'


**Vì sao sai:** cả ba ví dụ đề **trùng hợp** đúng (m = 1, 0, 3). `"abcd"` (m = 2) lộ lỗi.

### Lỗi 4 — Vòng lặp `i <= j` và đếm cả ký tự giữa

In [26]:
def wrong_middle(s):
    i, j = 0, len(s) - 1; cnt = 0
    while i <= j:
        cnt += s[i] != s[j] or i == j                            # SAI: coi ký tự giữa là "lệch"
        i, j = i + 1, j - 1
    return cnt <= 2

for s in ["abcdba", "abcde", "abcdz"]:
    print("bản sai :", wrong_middle(s), "| bản đúng:", Solution().makePalindrome(s), "cho", repr(s))

bản sai : True | bản đúng: True cho 'abcdba'
bản sai : False | bản đúng: True cho 'abcde'
bản sai : False | bản đúng: True cho 'abcdz'


**Vì sao sai:** `"abcde"` và `"abcdz"` có 2 cặp lệch nhưng bản sai đếm thêm ký tự giữa ⇒ 3 ⇒ `false`. `i <= j` tự nó không sai (khi `i == j` thì `s[i] == s[j]`), nhưng mọi xử lý đặc biệt cho ký tự giữa đều thừa. Ví dụ 1 (độ dài chẵn) **trùng hợp** đúng.

### Lỗi 5 — Dừng sớm với điều kiện lệch một (`cnt >= 2` thay vì `cnt > 2`)

In [27]:
def wrong_early(s):
    i, j = 0, len(s) - 1; cnt = 0
    while i < j:
        if s[i] != s[j]:
            cnt += 1
            if cnt >= 2: return False                             # SAI: 2 cặp lệch vẫn sửa được
        i += 1; j -= 1
    return True

for s in ["abcdba", "aa", "abcdef", "abcd", "abab"]:
    print("bản sai :", wrong_early(s), "| bản đúng:", Solution().makePalindrome(s), "cho", repr(s))

bản sai : True | bản đúng: True cho 'abcdba'
bản sai : True | bản đúng: True cho 'aa'
bản sai : False | bản đúng: False cho 'abcdef'
bản sai : False | bản đúng: True cho 'abcd'
bản sai : False | bản đúng: True cho 'abab'


**Vì sao sai:** cả ba ví dụ đề **trùng hợp** đúng (m = 1, 0, 3) — cùng một tập ví dụ không phân biệt được ngưỡng 1 và 2. `"abcd"`, `"abab"` (m = 2) lộ lỗi. Cách dừng sớm rất dễ lệch một khi viết vội; luôn thêm test có **đúng** `m = 2`.

**Ghi chú phụ — lát cắt `s[-h:]` với `h = 0`.** Ở cách 4, `s[-0:]` là **cả chuỗi** (vì `-0 == 0`). Trong `zip_halves` lỗi này bị che vì `s[:0]` rỗng nên `zip` không lấy phần tử nào, nhưng nếu dùng `s[-h:]` cho việc khác (so độ dài, ghép chuỗi) sẽ sai — an toàn hơn là `s[len(s) − h:]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Palindrome + "sửa ký tự" ⇒ đếm **cặp đối xứng lệch**; số thao tác tối thiểu = số cặp lệch.
- Palindrome + "xóa ký tự" ⇒ hai con trỏ + rẽ nhánh (LC 680) hoặc DP LCS với chuỗi đảo (LC 1216).
- Đọc kỹ "**đúng** k" vs "**tối đa** k": với "đúng", kiểm tra có thể "đốt" thao tác thừa mà không phá kết quả không.
- Hai con trỏ trên chuỗi: `while i < j`.
- Test tối thiểu cho mọi bài palindrome: chuỗi độ dài 1, độ dài chẵn/lẻ đã là palindrome, và chuỗi có số cặp lệch **đúng bằng** ngưỡng (ở đây `m = 2`) và **vượt** ngưỡng một đơn vị (`m = 3`).
- Bài cùng họ: 125 Valid Palindrome, 680 Valid Palindrome II, 1216 Valid Palindrome III, 1328 Break a Palindrome, 2697 Lexicographically Smallest Palindrome, 1400.

## 15. Biến thể của bài

### 15.1 Biến thể A — **Đúng `k`** thao tác (k bất kỳ)

Phân tích: một vị trí cần sửa dùng được `t ≥ 1` thao tác (đổi qua chữ trung gian); một vị trí không cần sửa chỉ được `t = 0` hoặc `t ≥ 2` (đổi đi rồi đổi về); một cặp khớp có thể đổi **cả hai** sang chữ mới (2 thao tác). Kết luận (kiểm chứng bằng BFS):
- `m ≥ 1`: được ⇔ `k ≥ m`.
- `m = 0`: được ⇔ `k = 0`, hoặc `k ≥ 2`, hoặc (`k = 1` và `n` lẻ).

In [28]:
def exactly_k(s, k):
    m = mismatches(s); n = len(s)
    if m >= 1: return k >= m
    return k == 0 or k >= 2 or (k == 1 and n % 2 == 1)

def exactly_k_brute(s, k):
    alpha = sorted(set(s) | {"x", "y"})
    cur = {s}
    for _ in range(k):
        cur = {t[:i] + c + t[i + 1:] for t in cur for i in range(len(t)) for c in alpha if c != t[i]}
    return any(t == t[::-1] for t in cur)

rng = random.Random(151)
for _ in range(300):
    n = rng.randint(1, 4); s = "".join(rng.choice("ab") for _ in range(n)); k = rng.randint(0, 4)
    assert exactly_k(s, k) == exactly_k_brute(s, k), (s, k)
print([exactly_k("aa", k) for k in range(5)], [exactly_k("aba", k) for k in range(5)], [exactly_k("abcd", k) for k in range(5)])

[True, False, True, True, True] [True, True, True, True, True] [False, False, True, True, True]


Bài gốc là `exactly_k(s, 1) or exactly_k(s, 2)` — với `m = 0` luôn đúng nhờ `k = 2`.

### 15.2 Biến thể B — Số thao tác **ít nhất** và palindrome **nhỏ nhất theo từ điển** (LC 2697)

In [29]:
def smallest_palindrome(s):
    t = list(s)
    for i in range(len(s) // 2):
        j = len(s) - 1 - i
        t[i] = t[j] = min(t[i], t[j])
    return mismatches(s), "".join(t)

print(smallest_palindrome("egcfe"), smallest_palindrome("abcd"), smallest_palindrome("seven"))

(1, 'efcfe') (2, 'abba') (1, 'neven')


### 15.3 Biến thể C — Được **xóa** tối đa một ký tự (LC 680)

In [30]:
def valid_palindrome_delete_one(s):
    def is_pal(i, j):
        while i < j:
            if s[i] != s[j]: return False
            i += 1; j -= 1
        return True
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]: return is_pal(i + 1, j) or is_pal(i, j - 1)
        i += 1; j -= 1
    return True

def delete_one_brute(s):
    return s == s[::-1] or any((t := s[:i] + s[i + 1:]) == t[::-1] for i in range(len(s)))

rng = random.Random(153)
for _ in range(1000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    assert valid_palindrome_delete_one(s) == delete_one_brute(s)
print(valid_palindrome_delete_one("abca"), valid_palindrome_delete_one("abc"))

True False


Khác bài gốc: xóa ký tự làm **dịch** các cặp đối xứng, nên không đếm cặp lệch độc lập được — phải thử hai nhánh tại cặp lệch đầu tiên.

### 15.4 Biến thể D — Mỗi thao tác chỉ đổi một chữ sang chữ **kế tiếp hoặc liền trước** (vòng tròn `a…z`), chi phí 1; tối thiểu tổng chi phí

Với cặp `(x, y)`, cần đưa hai ký tự về cùng một chữ `t`: chi phí `d(x, t) + d(t, y) ≥ d(x, y)` với `d` là khoảng cách vòng tròn, và đạt được khi `t` nằm trên cung ngắn. ⇒ đáp án `Σ d(sᵢ, s_{n−1−i})`.

In [31]:
def cyc(a, b):
    d = abs(ord(a) - ord(b)); return min(d, 26 - d)

def min_cyclic_cost(s):
    return sum(cyc(s[i], s[-1 - i]) for i in range(len(s) // 2))

def min_cyclic_brute(s):
    best = 0
    for i in range(len(s) // 2):
        x, y = s[i], s[-1 - i]
        best += min(cyc(x, t) + cyc(y, t) for t in string.ascii_lowercase)
    return best

rng = random.Random(154)
for _ in range(1000):
    s = "".join(rng.choice(string.ascii_lowercase) for _ in range(rng.randint(1, 10)))
    assert min_cyclic_cost(s) == min_cyclic_brute(s)
print(min_cyclic_cost("az"), min_cyclic_cost("abcdba"), min_cyclic_cost("abcdef"))

1 1 9


`"az"`: `a` và `z` kề nhau trên vòng tròn ⇒ chi phí 1. Brute ở đây thử mọi chữ đích `t` cho từng cặp (các cặp độc lập), xác nhận công thức khoảng cách vòng.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | định nghĩa "đúng 1 hoặc 2" | mũ theo n | lớn | kiểm chứng |
| 1b. So chuỗi đảo / 2 | đếm gấp đôi rồi chia | O(n) | O(n) | nhanh gọn |
| 2. Hai con trỏ | đếm cặp lệch | O(n) | O(1) | doocs |
| 3. Dừng sớm | `> 2` thì thoát | O(n) | O(1) | **mặc định** |
| 4. `zip` nửa | một dòng | O(n) | O(n) | cẩn thận `h = 0` |
| 5. Xây dựng nghiệm | trả thao tác | O(n) | O(n) | cần chỉ ra cách làm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `makePalindrome("abcba")`, `makePalindrome("abab")`, `makePalindrome("abcdcbaz")`?

**Bài 2 (tính tay):** Đếm cặp lệch của `"leetcode"` và cho đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(s)` trả về **số thao tác ít nhất** để `s` thành palindrome (cho phép 0).

**Bài 4 (tự viết lại):** Viết `bt4_todo(s)` cài bản nộp (dừng sớm) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(s, k)` trả về `True` ⇔ có thể thành palindrome với **tối đa** `k` thao tác **và** kết quả là palindrome nhỏ nhất theo từ điển có thể đạt được trong giới hạn đó — trả về chuỗi đó (hoặc `None`). Gợi ý: sửa các cặp lệch bằng chữ nhỏ hơn, rồi dùng thao tác dư để đổi các cặp `(x, x)` với `x > 'a'` thành `('a', 'a')` từ trái sang phải (mỗi cặp tốn 2), và ký tự giữa (tốn 1).

In [32]:
def bt3_todo(s):
    # TODO: số thao tác ít nhất
    pass

def bt4_todo(s):
    # TODO: bản nộp dừng sớm
    pass

def bt5_todo(s, k):
    # TODO: palindrome nhỏ nhất theo từ điển với tối đa k thao tác
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `"abcba"` là palindrome ⇒ `true`. Mức 2: `"abab"`: cặp `(a, b)`, `(b, a)` ⇒ 2 ⇒ `true`. Mức 3: `"abcdcbaz"`: cặp `(a,z), (b,a), (c,b), (d,c)` ⇒ 4 ⇒ `false`.

**Bài 2.** Mức 1: cặp `(l,e), (e,d), (e,o), (t,c)`. Mức 2: cả 4 lệch. Mức 3: `false`.

**Bài 3.** Mức 1: hai con trỏ. Mức 2: đếm cặp lệch. Mức 3: trả `m`.

**Bài 4.** Mức 1: `cnt = 0`. Mức 2: tăng khi lệch, thoát khi `> 2`. Mức 3: trả `True` cuối vòng.

**Bài 5.** Mức 1: nếu `m > k` ⇒ `None`. Mức 2: sửa mỗi cặp lệch thành `min` (tốn `m`); nếu chữ nhỏ hơn không phải `'a'` và còn dư 1 thao tác thì có thể đổi **cả hai** thành `'a'` (tốn thêm 1 cho cặp đó). Mức 3: duyệt từ trái: cặp khớp `x > 'a'` tốn 2 để thành `'a'`; cặp lệch tốn 1 (đã tính) + 1 nếu `min > 'a'`; ký tự giữa tốn 1 — tham lam từ trái vì vị trí trái quan trọng hơn trong thứ tự từ điển.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `true`, `true`, `false`. **Bài 2:** 4 cặp lệch ⇒ `false`.

In [33]:
assert Solution().makePalindrome("abcba") and Solution().makePalindrome("abab") and not Solution().makePalindrome("abcdcbaz")
assert mismatches("leetcode") == 4 and not Solution().makePalindrome("leetcode")

def bt3(s):
    return mismatches(s)

def bt4(s):
    return early_exit(s)

def bt5(s, k):
    n = len(s); m = mismatches(s)
    if m > k: return None
    t = list(s); budget = k - m
    for i in range(n // 2):
        j = n - 1 - i
        if t[i] != t[j]:
            c = min(t[i], t[j])
            if c != "a" and budget >= 1: c = "a"; budget -= 1     # đổi cả hai bên: thêm 1 thao tác
            t[i] = t[j] = c
        elif t[i] != "a" and budget >= 2:
            t[i] = t[j] = "a"; budget -= 2
    if n % 2 and t[n // 2] != "a" and budget >= 1:
        t[n // 2] = "a"
    return "".join(t)

def bt5_brute(s, k):
    alpha = sorted(set(s) | {"a"}); best = None
    for t in product(alpha, repeat=len(s)):
        t = "".join(t)
        if t == t[::-1] and sum(x != y for x, y in zip(s, t)) <= k:
            if best is None or t < best: best = t
    return best

rng = random.Random(2001)
for _ in range(400):
    n = rng.randint(1, 5); s = "".join(rng.choice("bcd") for _ in range(n)); k = rng.randint(0, 5)
    assert bt3(s) == min(sum(x != y for x, y in zip(s, t)) for t in ("".join(p) for p in product(sorted(set(s)), repeat=n)) if t == t[::-1])
    assert bt4(s) == brute(s)
    assert bt5(s, k) == bt5_brute(s, k), (s, k)
print(bt5("dcbd", 2), bt5("dcbd", 4), bt5("bcd", 1))
print("Đáp án bài tập: tất cả assert OK")

daad aaaa bcb
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
i, j, cnt = 0, len(s) - 1, 0
while i < j:
    if s[i] != s[j]:
        cnt += 1
        if cnt > 2: return False      # mỗi cặp lệch cần đúng 1 thao tác
    i += 1; j -= 1
return True                           # m = 0 vẫn đúng: đổi ký tự giữa / đổi cả một cặp
```

**Checklist:**
- [ ] Đếm **cặp** lệch (nửa chuỗi), không đếm vị trí lệch với chuỗi đảo.
- [ ] `m = 0` ⇒ `true` (chữ "đúng" vẫn thỏa).
- [ ] Ngưỡng là `≤ 2`, không phải `≤ 1`.
- [ ] (Premium) Đối chiếu lại đề gốc.